**Azure Virtual Network (VNet)** is the fundamental building block of private networking in Azure — the Azure equivalent of AWS VPC. A VNet is a logically isolated network in the Azure cloud where you deploy and connect Azure resources such as VMs, AKS clusters, App Service Environments, and private endpoints.

Every VNet is:
- **Region-scoped** — a VNet lives in one Azure region
- **Subscription-scoped** — a VNet belongs to one subscription
- **Free** — VNets themselves have no cost; you pay for gateways, peering, and certain connected resources

Resources inside a VNet can communicate privately. Traffic between VNets, or between a VNet and on-premises, requires peering, VPN, or ExpressRoute.

## Address Space and Subnets

### VNet address space

When you create a VNet you assign one or more **CIDR blocks** as the address space — the pool of IP addresses available for all subnets within the VNet. You can use any private RFC 1918 range:

| Range | Example CIDR |
|---|---|
| 10.0.0.0/8 | `10.0.0.0/16` — 65,536 addresses |
| 172.16.0.0/12 | `172.16.0.0/20` — 4,096 addresses |
| 192.168.0.0/16 | `192.168.0.0/24` — 256 addresses |

Plan your address space to avoid overlap with:
- Other VNets you plan to peer with
- On-premises networks connected via VPN or ExpressRoute

Overlapping address spaces prevent peering and routing.

### Subnets

A **subnet** is a subdivision of the VNet address space. Every resource you deploy into a VNet is placed in a subnet. Subnets:
- Must be non-overlapping within the VNet
- Can span all availability zones in the region (subnets are not AZ-scoped)
- Each subnet loses **5 IP addresses** to Azure reservations:

| Reserved address | Purpose |
|---|---|
| x.x.x.0 | Network address |
| x.x.x.1 | Default gateway |
| x.x.x.2 | Azure DNS mapping |
| x.x.x.3 | Azure DNS mapping |
| x.x.x.255 | Broadcast |

A `/28` subnet (16 addresses) gives you only **11 usable** addresses for resources.

### Subnet delegation

Some Azure services require a **dedicated subnet** with a delegation — for example:
- Azure App Service Environment → `Microsoft.Web/serverFarms`
- Azure SQL Managed Instance → `Microsoft.Sql/managedInstances`
- Azure Container Apps → `Microsoft.App/environments`
- Azure Bastion → must be named `AzureBastionSubnet`
- Azure Firewall → must be named `AzureFirewallSubnet`

Delegating a subnet grants the service permission to create service-specific network policies in that subnet and prevents other resource types from being deployed into it.

## Network Security Groups (NSGs)

A **Network Security Group** is a stateful packet filter — a list of **inbound** and **outbound** security rules that allow or deny traffic to and from Azure resources.

An NSG can be associated with:
- A **subnet** — rules apply to all resources in the subnet
- A **NIC (network interface)** — rules apply to that specific VM only

Both can be applied simultaneously. Azure evaluates **subnet NSG first**, then **NIC NSG**.

### NSG rule properties

| Property | Description |
|---|---|
| **Priority** | 100–4096; lower number = evaluated first |
| **Source / Destination** | IP address, CIDR range, Service Tag, or Application Security Group |
| **Protocol** | TCP, UDP, ICMP, ESP, AH, or Any |
| **Port range** | Single port, range, or `*` for any |
| **Action** | Allow or Deny |

Rules are evaluated in priority order — the **first matching rule wins**. If no rule matches, the default rules apply.

### Default NSG rules (always present, cannot be deleted)

**Inbound defaults:**

| Priority | Name | Source | Destination | Action |
|---|---|---|---|---|
| 65000 | AllowVnetInBound | VirtualNetwork | VirtualNetwork | Allow |
| 65001 | AllowAzureLoadBalancerInBound | AzureLoadBalancer | Any | Allow |
| 65500 | DenyAllInBound | Any | Any | Deny |

**Outbound defaults:**

| Priority | Name | Source | Destination | Action |
|---|---|---|---|---|
| 65000 | AllowVnetOutBound | VirtualNetwork | VirtualNetwork | Allow |
| 65001 | AllowInternetOutBound | Any | Internet | Allow |
| 65500 | DenyAllOutBound | Any | Any | Deny |

### Service Tags

**Service Tags** are named groups of IP address prefixes managed by Microsoft — they are updated automatically as Azure adds or changes service IPs. Use them instead of hardcoding IP ranges:

| Service Tag | Represents |
|---|---|
| `Internet` | Public IP space (outside VNet) |
| `VirtualNetwork` | VNet address space + peered VNets + on-prem routes |
| `AzureLoadBalancer` | Azure infrastructure load balancer |
| `Storage` | Azure Storage service IPs |
| `Sql` | Azure SQL service IPs |
| `AzureKeyVault` | Key Vault service IPs |
| `AppService` | App Service outbound IPs |

### Application Security Groups (ASGs)

**ASGs** let you group VMs by role (e.g. `web-servers`, `db-servers`) and write NSG rules that reference the group rather than IP addresses. As VMs are added or removed from the group, the rules apply automatically — no IP management required.

```
NSG Rule: Allow TCP 1433 from ASG:web-servers to ASG:db-servers
```

## Routing in Azure VNets

Azure automatically creates **system routes** for every subnet:

| Destination | Next hop | Purpose |
|---|---|---|
| VNet address space | Virtual Network | Intra-VNet traffic |
| 0.0.0.0/0 | Internet | Default internet egress |
| 10.0.0.0/8, 192.168.0.0/16, 172.16.0.0/12 | None | Drop RFC 1918 not in VNet |

### User-Defined Routes (UDRs)

**User-Defined Routes (UDRs)** override system routes — you create a **Route Table**, add custom routes, and associate it with a subnet.

Common uses:
- **Force-tunnel all internet traffic through Azure Firewall or an NVA** — add a `0.0.0.0/0 → VirtualAppliance` route
- **Route traffic between subnets through a firewall** — add specific prefix routes pointing to the firewall
- **Override BGP-propagated on-premises routes** — UDRs override routes learned from VPN/ExpressRoute gateways

### Next hop types

| Next hop type | Description |
|---|---|
| **Virtual Network** | Route within the VNet |
| **Internet** | Route to the public internet via Azure |
| **VirtualAppliance** | Route to a private IP (firewall NVA) |
| **VirtualNetworkGateway** | Route to VPN or ExpressRoute gateway |
| **None** | Drop the traffic (black hole) |

## DNS in Azure VNets

### Azure-provided DNS (default)

By default, VMs in a VNet resolve hostnames using **Azure-provided DNS** (168.63.129.16). It provides:
- Resolution of Azure resource hostnames within the VNet
- Resolution of public DNS names via Azure's recursive resolvers

Limitation: does not resolve custom domain names for resources in a VNet.

### Azure Private DNS Zones

**Azure Private DNS Zones** provide custom DNS resolution within a VNet — without deploying your own DNS servers.

```
Private zone: internal.contoso.com
  vm1.internal.contoso.com → 10.0.1.4
  db.internal.contoso.com  → 10.0.2.5
```

- Link the private DNS zone to a VNet (VNet link) — resources in the VNet can resolve records in the zone
- Enable **auto-registration** — VMs automatically get an A record when deployed into the linked VNet
- Private DNS zones are also used for **private endpoints** — when a private endpoint is created for a storage account, the FQDN resolves to the private IP via a private DNS zone (`privatelink.blob.core.windows.net`)

### Custom DNS servers

Set custom DNS server IPs on the VNet to redirect all DNS resolution to your own DNS infrastructure — for example, a Windows Server DNS server in the VNet or an on-premises DNS server reachable via ExpressRoute.

## Public IP Addresses and NAT

### Public IP addresses

| SKU | Allocation | Zone support | Use case |
|---|---|---|---|
| **Standard** | Static only | Zone-redundant or zonal | Load Balancer, Application Gateway, VPN Gateway, NAT Gateway, Bastion |
| **Basic** | Static or Dynamic | No | Legacy; being retired |

> Always use **Standard SKU** for new deployments — Basic is being deprecated.

### NAT Gateway

**Azure NAT Gateway** provides outbound-only internet connectivity for resources in a subnet — without needing a public IP on each VM.

- Attach one or more **Standard Public IPs** or **Public IP Prefixes** to the NAT Gateway
- Associate the NAT Gateway with a subnet — all outbound traffic from that subnet exits through the NAT Gateway's IPs
- **Fully managed** — no failover, no zone configuration needed
- Supports **64,512 SNAT ports per public IP** — eliminates SNAT port exhaustion issues seen with Load Balancer outbound rules

NAT Gateway is the recommended approach for predictable outbound IPs (e.g. for allowlisting) and for avoiding SNAT exhaustion in large-scale deployments.

## Azure Bastion

**Azure Bastion** is a managed PaaS service that provides secure RDP and SSH access to VMs directly from the Azure portal over TLS — without exposing VMs to the public internet.

```
Browser (HTTPS/443)  →  Azure Bastion  →  VM (RDP/3389 or SSH/22)
                        (in VNet)          (private IP only)
```

- Deploy Bastion into a dedicated subnet named **`AzureBastionSubnet`** with a `/26` or larger CIDR
- Bastion requires a **Standard Public IP**
- VMs do not need public IPs or NSG rules allowing RDP/SSH from the internet

### Bastion SKUs

| SKU | Max sessions | Native client support | Shareable link | IP-based connection |
|---|---|---|---|---|
| **Basic** | 2 concurrent sessions | No | No | No |
| **Standard** | 50+ sessions | Yes (RDP/SSH via local client) | Yes | Yes |
| **Premium** | 50+ sessions | Yes | Yes | Yes + session recording |

## Service Endpoints vs Private Endpoints

Both provide private access to Azure PaaS services from a VNet — but in different ways.

### Service Endpoints

A **service endpoint** extends the VNet's identity to the PaaS service. Traffic from the subnet routes over the Azure backbone (not the internet) to the service's **public IP**.

- Simple to enable — one click on the subnet
- The PaaS service still has a public endpoint; you lock it down with a firewall rule allowing only your subnet
- Source IP seen by the PaaS service is the private VNet IP
- Does **not** give the resource a private IP within the VNet

### Private Endpoints

A **private endpoint** creates a **network interface with a private IP** inside the VNet, mapped to a specific PaaS resource instance.

- The PaaS resource's FQDN resolves to the private IP via a private DNS zone
- Traffic never leaves the Azure backbone — fully private
- The public endpoint can be disabled entirely
- Accessible from peered VNets and on-premises (via VPN/ExpressRoute)

### Comparison

| | Service Endpoint | Private Endpoint |
|---|---|---|
| Private IP in VNet | No | Yes |
| FQDN resolves privately | No | Yes (private DNS zone) |
| Accessible from peered VNets | No | Yes |
| Accessible from on-premises | No | Yes |
| Disable public endpoint | No | Yes |
| Cost | Free | Per hour + data |
| Complexity | Low | Higher |

> **Private Endpoint** is the recommended approach for production workloads. **Service Endpoint** is a quick, low-cost option for VNet-only scenarios.

## VNet Integration for PaaS Services

**VNet Integration** is the outbound counterpart to Private Endpoints — it lets a PaaS service (App Service, Azure Functions, Container Apps) reach resources inside a VNet over private IPs.

```
App Service  →  [VNet Integration]  →  VNet subnet  →  private resources
(PaaS, no VNet)                         (outbound)      (VMs, private endpoints, etc.)
```

- VNet Integration is **outbound only** — it does not put the App Service inside the VNet for inbound traffic
- The App Service is assigned IPs from the integration subnet for outbound calls
- Combine with a **NAT Gateway** on the integration subnet for a stable outbound IP
- For **inbound** private access to an App Service, use a **Private Endpoint** on the App Service

| Direction | Mechanism |
|---|---|
| App Service → private VNet resources | **VNet Integration** |
| Private VNet clients → App Service | **Private Endpoint** on App Service |

## Working with VNets and NSGs via Python

In [ ]:
# pip install azure-identity azure-mgmt-network

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.network.models import (
    VirtualNetwork, AddressSpace, Subnet,
    NetworkSecurityGroup, SecurityRule,
    RouteTable, Route
)

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"
location        = "eastus"

net = NetworkManagementClient(credential, subscription_id)

In [ ]:
# Create a VNet with two subnets
vnet_params = VirtualNetwork(
    location=location,
    address_space=AddressSpace(address_prefixes=["10.0.0.0/16"]),
    subnets=[
        Subnet(name="web-subnet", address_prefix="10.0.1.0/24"),
        Subnet(name="db-subnet",  address_prefix="10.0.2.0/24"),
    ]
)
poller = net.virtual_networks.begin_create_or_update(resource_group, "my-vnet", vnet_params)
vnet   = poller.result()
print(f"VNet created: {vnet.name}  address space: {vnet.address_space.address_prefixes}")

# List VNets and their subnets
for v in net.virtual_networks.list(resource_group):
    print(f"\n{v.name} — {v.address_space.address_prefixes}")
    for s in (v.subnets or []):
        print(f"  subnet: {s.name:<20} {s.address_prefix}")

In [ ]:
# Create an NSG with an inbound rule allowing HTTPS from the internet
nsg_params = NetworkSecurityGroup(
    location=location,
    security_rules=[
        SecurityRule(
            name="Allow-HTTPS-Inbound",
            priority=100,
            direction="Inbound",
            access="Allow",
            protocol="Tcp",
            source_address_prefix="Internet",
            source_port_range="*",
            destination_address_prefix="*",
            destination_port_range="443"
        ),
        SecurityRule(
            name="Deny-All-Inbound",
            priority=4000,
            direction="Inbound",
            access="Deny",
            protocol="*",
            source_address_prefix="*",
            source_port_range="*",
            destination_address_prefix="*",
            destination_port_range="*"
        )
    ]
)
poller = net.network_security_groups.begin_create_or_update(resource_group, "web-nsg", nsg_params)
nsg    = poller.result()
print(f"NSG created: {nsg.name}")

In [ ]:
# Associate the NSG with the web-subnet
subnet_params = Subnet(
    address_prefix="10.0.1.0/24",
    network_security_group={"id": nsg.id}
)
poller = net.subnets.begin_create_or_update(resource_group, "my-vnet", "web-subnet", subnet_params)
subnet = poller.result()
print(f"NSG associated with subnet: {subnet.name}")

In [ ]:
# Create a route table that force-tunnels internet traffic through a firewall NVA
firewall_ip = "10.0.3.4"  # private IP of Azure Firewall or NVA

rt_params = RouteTable(
    location=location,
    routes=[
        Route(
            name="DefaultToFirewall",
            address_prefix="0.0.0.0/0",
            next_hop_type="VirtualAppliance",
            next_hop_ip_address=firewall_ip
        )
    ]
)
poller = net.route_tables.begin_create_or_update(resource_group, "web-rt", rt_params)
rt     = poller.result()
print(f"Route table created: {rt.name}")

# Attach the route table to the web-subnet
subnet_params = Subnet(
    address_prefix="10.0.1.0/24",
    network_security_group={"id": nsg.id},
    route_table={"id": rt.id}
)
poller = net.subnets.begin_create_or_update(resource_group, "my-vnet", "web-subnet", subnet_params)
subnet = poller.result()
print(f"Route table associated with subnet: {subnet.name}")

In [ ]:
# List effective routes on a NIC (useful for troubleshooting)
nic_name = "my-vm-nic"
poller = net.network_interfaces.begin_get_effective_route_table(resource_group, nic_name)
effective_routes = poller.result()

for route in effective_routes.value:
    prefixes = ", ".join(route.address_prefix or [])
    print(f"  {prefixes:<25} → {route.next_hop_type:<25} source: {route.source}")

## Summary

| Concept | Key Takeaway |
|---|---|
| VNet | Region-scoped, subscription-scoped private network; free to create |
| Address space | RFC 1918 CIDR block(s); plan for no overlap with peered VNets and on-premises |
| Subnet | Subdivision of VNet; loses 5 IPs to Azure; spans all AZs in region |
| Subnet delegation | Required for some PaaS services (ASE, SQL MI, Bastion, Firewall) |
| NSG | Stateful packet filter — inbound + outbound rules; applied to subnet and/or NIC |
| NSG priority | Lower number evaluated first; first match wins |
| Service Tags | Named IP groups managed by Microsoft — use instead of hardcoded IPs |
| ASGs | Group VMs by role; reference groups in NSG rules instead of IPs |
| System routes | Auto-created; handle intra-VNet and internet egress |
| UDRs | Override system routes — use to force-tunnel through a firewall |
| Azure-provided DNS | Default; resolves Azure hostnames and public DNS |
| Private DNS Zone | Custom DNS within a VNet; used by private endpoints; supports auto-registration |
| NAT Gateway | Managed outbound-only NAT; stable IPs; 64,512 SNAT ports per IP |
| Bastion | Managed RDP/SSH over HTTPS — no public IPs on VMs, no port 3389/22 exposure |
| Service Endpoint | Routes PaaS traffic over backbone; PaaS keeps public IP; VNet-only access |
| Private Endpoint | Private IP for PaaS in VNet; fully private; accessible from peered VNets and on-prem |
| VNet Integration | Outbound from App Service/Functions into a VNet subnet |
| Standard vs Basic Public IP | Always use Standard; Basic is being deprecated |